# Synthea

***Create 'raw' and 'extracted' folder inside the lakehouse to store zipped and extracted files**

In [1]:
import os
import re

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 3, Finished, Available, Finished, False)

In [2]:


# Define the folder where the original/raw Synthea files will be stored
raw_folder = "/lakehouse/default/Files/synthea/raw"

# Define the folder where compressed Synthea files will be extracted
extract_folder_clinical = "/lakehouse/default/Files/synthea/extracted/clinical/"
extract_folder_external = "/lakehouse/default/Files/synthea/extracted/external/"
extract_folder_external_atlast = "/lakehouse/default/Files/synthea/extracted/external/atlas/"
# Create the raw folder if it does not already exist
# exist_ok=True prevents an error if the folder already exists
os.makedirs(raw_folder, exist_ok=True)

# Create the extracted folder if it does not already exist
os.makedirs(extract_folder_clinical, exist_ok=True)
os.makedirs(extract_folder_external, exist_ok=True)
os.makedirs(extract_folder_external_atlast, exist_ok=True)


# Confirmation message
print("Folders ready")

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 4, Finished, Available, Finished, False)

Folders ready


### Download the compressed file

In [3]:
import requests

# Direct download URL for the ZIP file
url = "https://huggingface.co/datasets/ajitonelson/synthetic-mass-health-data/resolve/main/SyntheticMass_Data_Hack_ArangoDB.zip"

# Save location in the Lakehouse raw folder
download_path = f"{raw_folder}/synthea_data.zip"

# Download the file in chunks
response = requests.get(url, stream=True)
response.raise_for_status()

# Write downloaded content to the Lakehouse
with open(download_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print("Downloaded to:")
print(download_path)

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 5, Finished, Available, Finished, False)

Downloaded to:
/lakehouse/default/Files/synthea/raw/synthea_data.zip


### Extract the gzip 

In [4]:
import zipfile

# Check whether the downloaded file is a valid ZIP archive
if zipfile.is_zipfile(download_path):
    
    # Open the ZIP file
    with zipfile.ZipFile(download_path, "r") as zip_ref:
        
        # Extract all files into the target folder
        zip_ref.extractall(extract_folder_clinical)

    print("Files extracted to:")
    print(extract_folder_clinical)

else:
    print("Downloaded file is not a valid ZIP archive.")

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 6, Finished, Available, Finished, False)

Files extracted to:
/lakehouse/default/Files/synthea/extracted/clinical/


In [5]:
# Check what was extracted
for root, dirs, files in os.walk(extract_folder_clinical):
    for file in files:
           print(os.path.join(root, file))

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 7, Finished, Available, Finished, False)

/lakehouse/default/Files/synthea/extracted/clinical/encounters.csv
/lakehouse/default/Files/synthea/extracted/clinical/immunizations.csv
/lakehouse/default/Files/synthea/extracted/clinical/observations.csv
/lakehouse/default/Files/synthea/extracted/clinical/conditions.csv
/lakehouse/default/Files/synthea/extracted/clinical/medications.csv
/lakehouse/default/Files/synthea/extracted/clinical/careplans.csv
/lakehouse/default/Files/synthea/extracted/clinical/patients.csv
/lakehouse/default/Files/synthea/extracted/clinical/procedures.csv
/lakehouse/default/Files/synthea/extracted/clinical/allergies.csv
/lakehouse/default/Files/synthea/extracted/clinical/organizations.csv
/lakehouse/default/Files/synthea/extracted/clinical/payers.csv
/lakehouse/default/Files/synthea/extracted/clinical/providers.csv


### Test patients.csv file and schema

In [6]:
# patients_path = "Files/synthea/extracted/clinical/patients.csv"

# df_patients = (
#     spark.read
#         .option("header", "true")
#         .option("inferSchema", "true")
#         .csv(patients_path)
# )

# display(df_patients)

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 8, Finished, Available, Finished, False)

In [7]:
# print schema
# df_patients.printSchema()

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 9, Finished, Available, Finished, False)

In [8]:
# number of rows and coumns
# print(len(df_patients.columns), df_patients.count())

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 10, Finished, Available, Finished, False)

# Geo/Population Data

Get the following data:
1. County
2. Population
3. Median household income
4. Poverty rate
5. Insurance coverage
6. Urban/rural classification
7. Healthcare facility availability

#### USDA 2023 Rural-Urban Continuum Codes

In [9]:
# https://ers.usda.gov/data-products/atlas-of-rural-and-small-town-america/download-the-data?utm_source=chatgpt.com

import requests

files = [
    {
        "file_name": "rural_urban_2023.csv",
        "file_url": "https://www.ers.usda.gov/media/5768/2023-rural-urban-continuum-codes.csv?v=61279"
    },
    {
        "file_name": "ruralatlasdata.xlsx",
        "file_url": "https://ers.usda.gov/data-products/atlas-of-rural-and-small-town-america/download-the-data"
    }
]

for item in files:

    download_path = f"{raw_folder}/{item['file_name']}"

    response = requests.get(item["file_url"], stream=True)
    response.raise_for_status()

    with open(download_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print(f"Downloaded: {item['file_name']}")
    print(download_path)

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 11, Finished, Available, Finished, False)

Downloaded: rural_urban_2023.csv
/lakehouse/default/Files/synthea/raw/rural_urban_2023.csv
Downloaded: hospitals.csv
/lakehouse/default/Files/synthea/raw/hospitals.csv
Downloaded: ruralatlasdata.xlsx
/lakehouse/default/Files/synthea/raw/ruralatlasdata.xlsx


In [10]:
from pyspark.sql.functions import current_timestamp, lit

files = ["rural_urban_2023.csv"]

raw_folder = "Files/synthea/raw"
output_folder = "/lakehouse/default/Files/synthea/extracted/external"

for file in files:

    source_system = (
        "https://www.ers.usda.gov/media/5768/2023-rural-urban-continuum-codes.csv?v=61279"
        if file == "rural_urban_2023.csv"
        else
        "https://data.cms.gov/provider-data/sites/default/files/resources/893c372430d9d71a1c52737d01239d47_1785189955/Hospital_General_Information.csv"
    )

    file_path = f"{raw_folder}/{file}"

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(file_path)
    )

    df = (
        df
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_system", lit(source_system))
    )

    # Convert Spark DataFrame to Pandas since it is a small dataset
    pdf = df.toPandas()

    # Save directly as one CSV file
    pdf.to_csv(
        f"{output_folder}/{file}",
        index=False
    )

    display(df.limit(1))

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 00d75962-57a1-4247-9702-aafaa5d1be97)

Read the excel file and add ingestion metada

In [14]:
from datetime import datetime as t
import pandas as pd
excel_path = "https://ers.usda.gov/media/5775/download-the-data-in-excel.xlsx?v=67898"
extract_folder_external_atlast = "/lakehouse/default/Files/synthea/extracted/external/atlas/"

xls = pd.ExcelFile(
    excel_path,
    engine="openpyxl"
)

sheetnames= list(xls.sheet_names)
sheetnames

xdf= pd.DataFrame()

for sheet in sheetnames:
    xdf=pd.DataFrame()
    xdf = pd.read_excel( excel_path,sheet_name=sheet, engine="openpyxl")
    xdf['_ingestion_timestamp']= t.now()
    xdf['_source_system']='https://ers.usda.gov/media/5775/download-the-data-in-excel.xlsx?v=67898'
    
    xdf.to_csv(f"{extract_folder_external_atlast}{sheet}{'.csv'}", index=False)

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 15, Finished, Available, Finished, False)

# 

# Additional Synthea data
1. Organizations
2. Payers
3. Providers

In [17]:
import os
import requests
import pandas as pd
from datetime import datetime, timezone

base_url = "https://huggingface.co/datasets/VRKomari/100K_tx_synthetic_patient_data/resolve/main"

files = [
    "organizations.parquet",
    "providers.parquet",
    "payers.parquet"
]

raw_folder = "/lakehouse/default/Files/synthea/raw"
clinical_folder = "/lakehouse/default/Files/synthea/extracted/clinical"

os.makedirs(raw_folder, exist_ok=True)
os.makedirs(clinical_folder, exist_ok=True)

for file in files:

    # -----------------------------------
    # 1. Download original Parquet to RAW
    # -----------------------------------
    url = f"{base_url}/{file}"
    parquet_path = f"{raw_folder}/{file}"

    print(f"Downloading: {file}")

    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(parquet_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print(f"Downloaded to: {parquet_path}")

    # -----------------------------------
    # 2. Read Parquet
    # -----------------------------------
    df = pd.read_parquet(parquet_path)

    # -----------------------------------
    # 3. Add ingestion metadata
    # -----------------------------------
    df["_ingestion_timestamp"] = datetime.now(timezone.utc)
    df["_source_system"] = "HuggingFace_Synthea"
    df["_source_file"] = file

    # -----------------------------------
    # 4. Convert and save as CSV
    # -----------------------------------
    csv_file = file.replace(".parquet", ".csv")
    csv_path = f"{clinical_folder}/{csv_file}"

    df.to_csv(
        csv_path,
        index=False
    )

    print(f"Saved: {csv_path}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print("-" * 60)

StatementMeta(, f8c79345-4c75-4941-9083-2dedc76fd462, 18, Finished, Available, Finished, False)

Downloading: organizations.parquet
Downloaded to: /lakehouse/default/Files/synthea/raw/organizations.parquet
Saved: /lakehouse/default/Files/synthea/extracted/clinical/organizations.csv
Rows: 6,888
Columns: 14
------------------------------------------------------------
Downloading: providers.parquet
Downloaded to: /lakehouse/default/Files/synthea/raw/providers.parquet


Saved: /lakehouse/default/Files/synthea/extracted/clinical/providers.csv
Rows: 6,888
Columns: 16
------------------------------------------------------------
Downloading: payers.parquet
Downloaded to: /lakehouse/default/Files/synthea/raw/payers.parquet
Saved: /lakehouse/default/Files/synthea/extracted/clinical/payers.csv
Rows: 10
Columns: 25
------------------------------------------------------------
